# CleanDIFT: Diffusion Features without Noise

This notebook serves as a demonstration of the CleanDIFT fine-tuning appraoch to obtain clean diffusion features from standard diffusion models without noise.

## Background: Why CleanDIFT?

### The Problem with Traditional Diffusion Features (DIFT)

Diffusion models like Stable Diffusion have been shown to lean powerful semantic representations. However, extracting these features traditionally requires:

1. **Adding noise to clean images** - destroy information
2. **Choosing a noise level (timestep)** - task-dependent hyperparameter
3. **Multiple forward passes** - ensemble over timesteps for robustness

### The CleanDIFT Solution

CleanDIFT uses a lightweight, unsupervised fine-tuning approach that:

- Enable the diffusion model to work directly with **clean images** (no noise)
- Produces **timestep-independent features** (no hyperparamenter tuning)
- Requires only a **single forward pass** (50x faster)

### Training Details

- **Duration:** 30 minutes on a single GPU
- **Dataset:** Random subset of COYO-700M (images ≥ 512×512)
- **Optimizer:** Adam with batch size 8, learning rate $2^{-6}$
- **Noise levels:** Stratified sampling of 3 different noise levels per image
- **Backbone:** Stable Diffusion 1.5 or 2.1

## Setups and Imports

### Packages Installation

In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install diffusers transformers accelerate scipy safetensors ftfy bitsandbytes einops opencv-python-headless matplotlib

In [18]:
%pip install jaxtyping omegaconf

Note: you may need to restart the kernel to use updated packages.


### Package Imports

In [ ]:
import sys
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from safetensors.torch import load_file
from torchvision.transforms.functional import to_tensor, to_pil_image
from huggingface_hub import hf_hub_download
from diffusers import UNet2DConditionModel, AutoencoderKL
from transformers import CLIPTextModel, CLIPTokenizer
from omegaconf import OmegaConf

print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Understanding CleanDIFT Architecture

Before loading pretrained models, let's understand and reimplement the CleanDIFT architecture step-by-step.

### Architecture Overview

CleanDIFT consists of three main components:

1. **Frozen Stable Diffusion Backbone** (SD 1.5 or SD 2.1)
   - UNet with attention mechanisms
   - VAE for latent encoding/decoding
   - CLIP text encoder for conditioning

2. **Learnable Projection Heads** (FFN Stacks)
   - One projection head per feature layer
   - 3 stacked Feed-Forward Networks (FFNs)
   - Zero-initialized residual connections
   - Act as identity mapping initially

3. **Timestep Mapping Network**
   - Learns to map timestep embeddings
   - Allows model to work at t=0 (clean images)
   - Small MLP with 2-3 layers

### Training Process
- Fine-tune only projection heads and mapping network
- Keep SD backbone frozen (no gradients)
- Unsupervised learning on COYO-700M random subset
- 30 minutes on single GPU

## Implementation

### Imports

In [4]:
import math
import numpy as np
import random
import torch
import torch.nn as nn
from dataclasses import dataclass
from typing import Dict, Union, Any
import torch.nn.functional as F
from functools import reduce

### Utilities

Here we define some utilities to:

- Set the seed for experiments with a default value of 42 using the `set_seed()` function to ensure:
    - Reproducibility
    - Variability and diversity
    - Fair experimentation
- Move PyTorch tensors in a dictionary to a specific device (like CPU and GPU) while preserving non-tensor values unchanged using `dict_to()`
- Initialize both **weights** and **biases** (if present) of a layer to zero.
    - Used for **residual connections** to ensure stable training at initialization.
- Define the architecture parameters using a _dataclass_ for the **Timestep Mapping Network**:
    - **Depth**: Number of feedforward layers in the mapping network (typically 2)
    - **width**: Hidden dimension size (typically 256)
    - **d_ff**: Feedforward expansion dimension (typically 768)
    - **droupout**: Dropout rate for regularization (typically 0.0)

The **Timestep Mapping Network** is one of the key innovations which:
1. Takes timestep embeddings as input (via `FourierFeatures`)
2. Processes them through a small MLP (`MappingNetwork`)
3. Produces conditioning vectors (smaller size vectors) for the **projection heads**
    - Allows CleanDIFT to work at $t = 0$ for clean images.
    - ***The mapping network learn to produce appropriate conditioning even when the actual timestep is zero***.

In [5]:
def set_seed(seed=42, cuda=True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if cuda:
        torch.cuda.manual_seed_all(seed)


def dict_to(
    d: Dict[str, Union[torch.Tensor, Any]], **to_kwargs
) -> Dict[str, Union[torch.Tensor, Any]]:
    return {
        k: (v.to(**to_kwargs) if isinstance(v, torch.Tensor) else v)
        for k, v in d.items()
    }


# Helpers
def zero_init(layer):
    nn.init.zeros_(layer.weight)
    if layer.bias is not None:
        nn.init.zeros_(layer.bias)
    return layer


@dataclass
class MappingSpec:
    depth: int
    width: int
    d_ff: int
    dropout: float

### Feed Forward Network

Here we define the `FeedForwardBlock` which is: 

- Responsible for processing the timestep embeddings in the `MappingNetwork`. It is composed of:
    - Stack of 2 FFN blocks per feature layer (`depth=2`)
    - Transforms timestep information into conditioning vectors
    - No conditional normalization (standard `RMSNorm`)
- A core component of the _learnable_ projection heads (Adapters in `FFNStack`)
    - Stack of 3 FFN blocks per feature layer (`depth=3`) 
    - **Conditioned on timestep embeddings** via `AdaRMSNorm` (Adaptive Normalization)
    - Projects frozen Stable Diffusion features into task specific representations

Additionally we have:
- `RMSNorm` which regularize the summed inputs to a neuron in one layer according to the _root mean square_ (RMS), giving the model re-scaling invariance property and implicit learning rate adaptation ability.
    - Simpler and more efficient than `LayerNorm`
    - Used:
        - In `FeedForwardBlock` for standard normalization
        - In `MappingNetwork` for I/O normalization
        - As the base for `AdaRMSNorm`
    - Stabilizes training by:
        - Normalizing activation magnitudes to prevent exploding/vanishing gradients
        - The learnable `scale` parameter allows the model to control normalization strength per feature
        - $eps=1e^{-6}$ prevents division by zero
- `MappingNetwork` which enables the model to work with clean images ($t = 0$) instead of requiring noisy inputs.
    1. Learn a timestep representation space where $t = 0$ is meaningful
    2. Produce a conditioning information that tells the adapters _how to behave_ even at $t = 0$
    3. Bridge the gap between training (various noise levels) and inference (clean images)

        ```graph
        Input: timestep embedding (256-dim from FourierFeatures)
            ↓
        RMSNorm (in_norm)
            ↓
        FeedForwardBlock #1 (256 → 768 → 256)
            ↓
        FeedForwardBlock #2 (256 → 768 → 256)
            ↓
        RMSNorm (out_norm)
            ↓
        Output: conditioning vector (256-dim)
        ```
    - Output is passed to `AdaRMSNorm` in the projection heads (`FFNStack`)
        - Allow the backbone to produce meaningfull features at $t = 0$
    - Reason why CleanDIFT can extract features 50x faster (single forward pass at ($t = 0$) than DIFT.
- `LinearSwiGLU` used in `FeedForwardBlock` as the `up_proj` layer:
    - Gating mechanism that allows selective information flow
    - Consistent improvement over standard FFNs
    - Smooth gradient which helps optimization
    - Helps the network lean **context-dependent activations**
        - Selectively refine different aspects of the frozen diffusion features based on the task.
- `Linear` which works with `zero_init` function to directly access weight and bias attributes.
- `FourierFeatures` which implements **Random Fourier Features** (RFF) for encoding continuous scalar values (like timesteps) into high-dimensional sinusoidal representations.
    - **Capture periodicity**: Timesteps have periodic patterns
    - **High-frequency details**: Multiple random frequencies capture different temporal scales
    - **Neural Tangent Kernel approximation**: RFF approximates infinite-dimensional kernel methods
    - **Better than simple embedding**: Continous values need smooth, expressive representations
    - _Example_: A single number can be mapped into 256 real values using it.
    - Here, ***rich representation helps the mapping network distinguish between different timesteps and lean appropriate conditioning for each***.

In [7]:
class FeedForwardBlock(nn.Module):
    def __init__(self, d_model, d_ff, d_cond_norm=None, dropout=0.0):
        super().__init__()
        if d_cond_norm is not None:
            self.norm = AdaRMSNorm(d_model, d_cond_norm)
        else:
            self.norm = RMSNorm(d_model)
        self.up_proj = LinearSwiGLU(d_model, d_ff, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.down_proj = zero_init(Linear(d_ff, d_model, bias=False))

    def forward(self, x, cond_norm=None, **kwargs):
        skip = x
        if cond_norm is not None:
            x = self.norm(x, cond_norm)
        else:
            x = self.norm(x)
        x = self.up_proj(x)
        x = self.dropout(x)
        x = self.down_proj(x)
        return x + skip


class RMSNorm(nn.Module):
    def __init__(self, shape, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(shape))

    def extra_repr(self):
        return f"shape={tuple(self.scale.shape)}, eps={self.eps}"

    def forward(self, x):
        return rms_norm(x, self.scale, self.eps)


def rms_norm(x, scale, eps):
    dtype = reduce(torch.promote_types, (x.dtype, scale.dtype, torch.float32))
    mean_sq = torch.mean(x.to(dtype) ** 2, dim=-1, keepdim=True)
    scale = scale.to(dtype) * torch.rsqrt(mean_sq + eps)
    return x * scale.to(x.dtype)


class AdaRMSNorm(nn.Module):
    def __init__(self, features, cond_features, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.linear = zero_init(Linear(cond_features, features, bias=False))

    def extra_repr(self):
        return f"eps={self.eps},"

    def forward(self, x, cond):
        return rms_norm(x, self.linear(cond)[:, None, :] + 1, self.eps)


class MappingNetwork(nn.Module):
    def __init__(self, n_layers, d_model, d_ff, dropout=0.0):
        super().__init__()
        self.in_norm = RMSNorm(d_model)
        self.blocks = nn.ModuleList(
            [FeedForwardBlock(d_model, d_ff, dropout=dropout) for _ in range(n_layers)]
        )
        self.out_norm = RMSNorm(d_model)

    def forward(self, x):
        x = self.in_norm(x)
        for block in self.blocks:
            x = block(x)
        x = self.out_norm(x)
        return x


def linear_swiglu(x, weight, bias=None):
    x = x @ weight.mT
    if bias is not None:
        x = x + bias
    x, gate = x.chunk(2, dim=-1)
    return x * F.silu(gate)


class LinearSwiGLU(nn.Linear):
    def __init__(self, in_features, out_features, bias=True):
        super().__init__(in_features, out_features * 2, bias=bias)
        self.out_features = out_features

    def forward(self, x):
        return linear_swiglu(x, self.weight, self.bias)


class Linear(nn.Linear):
    def forward(self, x):
        return super().forward(x)


class FourierFeatures(nn.Module):
    def __init__(self, in_features, out_features, std=1.0):
        super().__init__()
        assert out_features % 2 == 0
        self.register_buffer(
            "weight", torch.randn([out_features // 2, in_features]) * std
        )

    def forward(self, input):
        f = 2 * math.pi * input @ self.weight.T
        return torch.cat([f.cos(), f.sin()], dim=-1)

### Minimal Stable Diffusion 2.1

In [8]:
# Obviously modified from the original source code
# https://github.com/huggingface/diffusers
# So has APACHE 2.0 license

# Author : Simo Ryu
# Adapted for SD21 by Nick Stracke

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

from collections import namedtuple


class Timesteps(nn.Module):
    def __init__(self, num_channels: int = 320):
        super().__init__()
        self.num_channels = num_channels

    def forward(self, timesteps):
        half_dim = self.num_channels // 2
        exponent = -math.log(10000) * torch.arange(
            half_dim, dtype=torch.float32, device=timesteps.device
        )
        exponent = exponent / (half_dim - 0.0)

        emb = torch.exp(exponent)
        emb = timesteps[:, None].float() * emb[None, :]

        sin_emb = torch.sin(emb)
        cos_emb = torch.cos(emb)
        emb = torch.cat([cos_emb, sin_emb], dim=-1)

        return emb


class TimestepEmbedding(nn.Module):
    def __init__(self, in_features, out_features):
        super(TimestepEmbedding, self).__init__()
        self.linear_1 = nn.Linear(in_features, out_features, bias=True)
        self.act = nn.SiLU()
        self.linear_2 = nn.Linear(out_features, out_features, bias=True)

    def forward(self, sample):
        sample = self.linear_1(sample)
        sample = self.act(sample)
        sample = self.linear_2(sample)

        return sample


class ResnetBlock2D(nn.Module):
    def __init__(self, in_channels, out_channels, conv_shortcut=True):
        super(ResnetBlock2D, self).__init__()
        self.norm1 = nn.GroupNorm(32, in_channels, eps=1e-05, affine=True)
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, kernel_size=3, stride=1, padding=1
        )
        self.time_emb_proj = nn.Linear(1280, out_channels, bias=True)
        self.norm2 = nn.GroupNorm(32, out_channels, eps=1e-05, affine=True)
        self.dropout = nn.Dropout(p=0.0, inplace=False)
        self.conv2 = nn.Conv2d(
            out_channels, out_channels, kernel_size=3, stride=1, padding=1
        )
        self.nonlinearity = nn.SiLU()
        self.conv_shortcut = None
        if conv_shortcut:
            self.conv_shortcut = nn.Conv2d(
                in_channels, out_channels, kernel_size=1, stride=1
            )

    def forward(self, input_tensor, temb):
        hidden_states = input_tensor
        hidden_states = self.norm1(hidden_states)
        hidden_states = self.nonlinearity(hidden_states)

        hidden_states = self.conv1(hidden_states)

        temb = self.nonlinearity(temb)
        temb = self.time_emb_proj(temb)[:, :, None, None]
        hidden_states = hidden_states + temb
        hidden_states = self.norm2(hidden_states)

        hidden_states = self.nonlinearity(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.conv2(hidden_states)

        if self.conv_shortcut is not None:
            input_tensor = self.conv_shortcut(input_tensor)

        output_tensor = input_tensor + hidden_states

        return output_tensor


class Attention(nn.Module):
    def __init__(
        self, inner_dim, cross_attention_dim=None, num_heads=None, dropout=0.0
    ):
        super(Attention, self).__init__()
        if num_heads is None:
            self.head_dim = 64
            self.num_heads = inner_dim // self.head_dim
        else:
            self.num_heads = num_heads
            self.head_dim = inner_dim // num_heads

        self.scale = self.head_dim**-0.5
        if cross_attention_dim is None:
            cross_attention_dim = inner_dim
        self.to_q = nn.Linear(inner_dim, inner_dim, bias=False)
        self.to_k = nn.Linear(cross_attention_dim, inner_dim, bias=False)
        self.to_v = nn.Linear(cross_attention_dim, inner_dim, bias=False)

        self.to_out = nn.ModuleList(
            [nn.Linear(inner_dim, inner_dim), nn.Dropout(dropout, inplace=False)]
        )

    def forward(self, hidden_states, encoder_hidden_states=None):
        q = self.to_q(hidden_states)
        k = (
            self.to_k(encoder_hidden_states)
            if encoder_hidden_states is not None
            else self.to_k(hidden_states)
        )
        v = (
            self.to_v(encoder_hidden_states)
            if encoder_hidden_states is not None
            else self.to_v(hidden_states)
        )

        # # NOTE SD21 upcasts the attention computions
        # I checked and not upcasting in bfloat16 basically leads to no visible differences ~Nick
        # dtype = q.dtype
        # q = q.float()
        # k = k.float()
        # # typically v is not upcasted but we need to do it to work with sdpa
        # v = v.float()

        b, t, c = q.size()

        q = q.view(q.size(0), q.size(1), self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(k.size(0), k.size(1), self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(v.size(0), v.size(1), self.num_heads, self.head_dim).transpose(1, 2)

        attn_output = F.scaled_dot_product_attention(
            q, k, v, dropout_p=0.0, scale=self.scale
        )
        # attn_output = attn_output.to(dtype)
        attn_output = attn_output.transpose(1, 2).contiguous().view(b, t, c)

        for layer in self.to_out:
            attn_output = layer(attn_output)

        return attn_output


class GEGLU(nn.Module):
    def __init__(self, in_features, out_features):
        super(GEGLU, self).__init__()
        self.proj = nn.Linear(in_features, out_features * 2, bias=True)

    def forward(self, x):
        x_proj = self.proj(x)
        x1, x2 = x_proj.chunk(2, dim=-1)
        return x1 * torch.nn.functional.gelu(x2)


class FeedForward(nn.Module):
    def __init__(self, in_features, out_features):
        super(FeedForward, self).__init__()

        self.net = nn.ModuleList(
            [
                GEGLU(in_features, out_features * 4),
                nn.Dropout(p=0.0, inplace=False),
                nn.Linear(out_features * 4, out_features, bias=True),
            ]
        )

    def forward(self, x):
        for layer in self.net:
            x = layer(x)
        return x


class BasicTransformerBlock(nn.Module):
    def __init__(self, hidden_size):
        super(BasicTransformerBlock, self).__init__()
        self.norm1 = nn.LayerNorm(hidden_size, eps=1e-05, elementwise_affine=True)
        self.attn1 = Attention(hidden_size)
        self.norm2 = nn.LayerNorm(hidden_size, eps=1e-05, elementwise_affine=True)
        self.attn2 = Attention(hidden_size, 1024)
        self.norm3 = nn.LayerNorm(hidden_size, eps=1e-05, elementwise_affine=True)
        self.ff = FeedForward(hidden_size, hidden_size)

    def forward(self, x, encoder_hidden_states=None):
        residual = x

        x = self.norm1(x)
        x = self.attn1(x)
        x = x + residual

        residual = x

        x = self.norm2(x)
        if encoder_hidden_states is not None:
            x = self.attn2(x, encoder_hidden_states)
        else:
            x = self.attn2(x)
        x = x + residual

        residual = x

        x = self.norm3(x)
        x = self.ff(x)
        x = x + residual
        return x


class Transformer2DModel(nn.Module):
    def __init__(self, in_channels, out_channels, n_layers):
        super(Transformer2DModel, self).__init__()
        self.norm = nn.GroupNorm(32, in_channels, eps=1e-06, affine=True)
        self.proj_in = nn.Linear(in_channels, out_channels, bias=True)
        self.transformer_blocks = nn.ModuleList(
            [BasicTransformerBlock(out_channels) for _ in range(n_layers)]
        )
        self.proj_out = nn.Linear(out_channels, out_channels, bias=True)

    def forward(self, hidden_states, encoder_hidden_states=None):
        batch, _, height, width = hidden_states.shape
        res = hidden_states
        hidden_states = self.norm(hidden_states)
        inner_dim = hidden_states.shape[1]
        hidden_states = hidden_states.permute(0, 2, 3, 1).reshape(
            batch, height * width, inner_dim
        )
        hidden_states = self.proj_in(hidden_states)

        for block in self.transformer_blocks:
            hidden_states = block(hidden_states, encoder_hidden_states)

        hidden_states = self.proj_out(hidden_states)
        hidden_states = (
            hidden_states.reshape(batch, height, width, inner_dim)
            .permute(0, 3, 1, 2)
            .contiguous()
        )

        return hidden_states + res


class Downsample2D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(Downsample2D, self).__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels, kernel_size=3, stride=2, padding=1
        )

    def forward(self, x):
        return self.conv(x)


class Upsample2D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(Upsample2D, self).__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels, kernel_size=3, stride=1, padding=1
        )

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2.0, mode="nearest")
        return self.conv(x)


class DownBlock2D(nn.Module):
    def __init__(self, in_channels, out_channels, has_downsamplers=True):
        super(DownBlock2D, self).__init__()
        self.resnets = nn.ModuleList(
            [
                ResnetBlock2D(in_channels, out_channels, conv_shortcut=False),
                ResnetBlock2D(out_channels, out_channels, conv_shortcut=False),
            ]
        )
        self.downsamplers = None
        if has_downsamplers:
            self.downsamplers = nn.ModuleList(
                [Downsample2D(out_channels, out_channels)]
            )

    def forward(self, hidden_states, temb):
        output_states = []
        for module in self.resnets:
            hidden_states = module(hidden_states, temb)
            output_states.append(hidden_states)

        if self.downsamplers is not None:
            hidden_states = self.downsamplers[0](hidden_states)
            output_states.append(hidden_states)

        return hidden_states, output_states


class CrossAttnDownBlock2D(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        n_layers,
        has_downsamplers=True,
        conv_shortcut=True,
    ):
        super(CrossAttnDownBlock2D, self).__init__()
        self.attentions = nn.ModuleList(
            [
                Transformer2DModel(out_channels, out_channels, n_layers),
                Transformer2DModel(out_channels, out_channels, n_layers),
            ]
        )
        self.resnets = nn.ModuleList(
            [
                ResnetBlock2D(in_channels, out_channels, conv_shortcut),
                ResnetBlock2D(out_channels, out_channels, conv_shortcut=False),
            ]
        )
        self.downsamplers = None
        if has_downsamplers:
            self.downsamplers = nn.ModuleList(
                [Downsample2D(out_channels, out_channels)]
            )

    def forward(self, hidden_states, temb, encoder_hidden_states):
        output_states = []
        for resnet, attn in zip(self.resnets, self.attentions):
            hidden_states = resnet(hidden_states, temb)
            hidden_states = attn(
                hidden_states,
                encoder_hidden_states=encoder_hidden_states,
            )
            output_states.append(hidden_states)

        if self.downsamplers is not None:
            hidden_states = self.downsamplers[0](hidden_states)
            output_states.append(hidden_states)

        return hidden_states, output_states


class CrossAttnUpBlock2D(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        prev_output_channel,
        n_layers,
        has_upsamplers=True,
    ):
        super(CrossAttnUpBlock2D, self).__init__()
        self.attentions = nn.ModuleList(
            [
                Transformer2DModel(out_channels, out_channels, n_layers),
                Transformer2DModel(out_channels, out_channels, n_layers),
                Transformer2DModel(out_channels, out_channels, n_layers),
            ]
        )
        self.resnets = nn.ModuleList(
            [
                ResnetBlock2D(prev_output_channel + out_channels, out_channels),
                ResnetBlock2D(2 * out_channels, out_channels),
                ResnetBlock2D(out_channels + in_channels, out_channels),
            ]
        )

        self.upsamplers = None
        if has_upsamplers:
            self.upsamplers = nn.ModuleList([Upsample2D(out_channels, out_channels)])

    def forward(
        self, hidden_states, res_hidden_states_tuple, temb, encoder_hidden_states
    ):
        all_hidden_states = []
        for resnet, attn in zip(self.resnets, self.attentions):
            # pop res hidden states
            res_hidden_states = res_hidden_states_tuple[-1]
            res_hidden_states_tuple = res_hidden_states_tuple[:-1]
            hidden_states = torch.cat([hidden_states, res_hidden_states], dim=1)
            hidden_states = resnet(hidden_states, temb)
            hidden_states = attn(
                hidden_states,
                encoder_hidden_states=encoder_hidden_states,
            )
            all_hidden_states.append(hidden_states)

        if self.upsamplers is not None:
            for upsampler in self.upsamplers:
                hidden_states = upsampler(hidden_states)
                all_hidden_states[-1] = hidden_states

        return hidden_states, all_hidden_states


class UpBlock2D(nn.Module):
    def __init__(
        self, in_channels, out_channels, prev_output_channel, has_upsamplers=True
    ):
        super(UpBlock2D, self).__init__()
        self.resnets = nn.ModuleList(
            [
                ResnetBlock2D(out_channels + prev_output_channel, out_channels),
                ResnetBlock2D(out_channels * 2, out_channels),
                ResnetBlock2D(out_channels + in_channels, out_channels),
            ]
        )

        self.upsamplers = None
        if has_upsamplers:
            self.upsamplers = nn.ModuleList([Upsample2D(out_channels, out_channels)])

    def forward(self, hidden_states, res_hidden_states_tuple, temb=None):
        all_hidden_states = []
        for resnet in self.resnets:
            res_hidden_states = res_hidden_states_tuple[-1]
            res_hidden_states_tuple = res_hidden_states_tuple[:-1]
            hidden_states = torch.cat([hidden_states, res_hidden_states], dim=1)
            hidden_states = resnet(hidden_states, temb)
            all_hidden_states.append(hidden_states)

        if self.upsamplers is not None:
            for upsampler in self.upsamplers:
                hidden_states = upsampler(hidden_states)
                all_hidden_states[-1] = hidden_states

        return hidden_states, all_hidden_states


class UNetMidBlock2DCrossAttn(nn.Module):
    def __init__(self, in_features):
        super(UNetMidBlock2DCrossAttn, self).__init__()
        self.attentions = nn.ModuleList(
            [Transformer2DModel(in_features, in_features, n_layers=1)]
        )
        self.resnets = nn.ModuleList(
            [
                ResnetBlock2D(in_features, in_features, conv_shortcut=False),
                ResnetBlock2D(in_features, in_features, conv_shortcut=False),
            ]
        )

    def forward(self, hidden_states, temb=None, encoder_hidden_states=None):
        hidden_states = self.resnets[0](hidden_states, temb)
        for attn, resnet in zip(self.attentions, self.resnets[1:]):
            hidden_states = attn(
                hidden_states,
                encoder_hidden_states=encoder_hidden_states,
            )
            hidden_states = resnet(hidden_states, temb)

        return hidden_states


class SD21UNetModel(nn.Module):
    def __init__(self):
        super(SD21UNetModel, self).__init__()

        # This is needed to imitate huggingface config behavior
        # has nothing to do with the model itself
        # remove this if you don't use diffuser's pipeline
        self.config = namedtuple("config", "in_channels time_cond_proj_dim sample_size")
        self.config.in_channels = 4
        # self.config.addition_time_embed_dim = 256
        self.config.sample_size = 96
        self.config.time_cond_proj_dim = None

        self.conv_in = nn.Conv2d(4, 320, kernel_size=3, stride=1, padding=1)
        self.time_proj = Timesteps()
        self.time_embedding = TimestepEmbedding(in_features=320, out_features=1280)
        self.down_blocks = nn.ModuleList(
            [
                CrossAttnDownBlock2D(
                    in_channels=320, out_channels=320, n_layers=1, conv_shortcut=False
                ),
                CrossAttnDownBlock2D(in_channels=320, out_channels=640, n_layers=1),
                CrossAttnDownBlock2D(in_channels=640, out_channels=1280, n_layers=1),
                DownBlock2D(
                    in_channels=1280, out_channels=1280, has_downsamplers=False
                ),
            ]
        )
        self.up_blocks = nn.ModuleList(
            [
                UpBlock2D(
                    in_channels=1280, out_channels=1280, prev_output_channel=1280
                ),
                CrossAttnUpBlock2D(
                    in_channels=640,
                    out_channels=1280,
                    prev_output_channel=1280,
                    n_layers=1,
                ),
                CrossAttnUpBlock2D(
                    in_channels=320,
                    out_channels=640,
                    prev_output_channel=1280,
                    n_layers=1,
                ),
                CrossAttnUpBlock2D(
                    in_channels=320,
                    out_channels=320,
                    prev_output_channel=640,
                    n_layers=1,
                    has_upsamplers=False,
                ),
            ]
        )

        self.mid_block = UNetMidBlock2DCrossAttn(1280)
        self.conv_norm_out = nn.GroupNorm(32, 320, eps=1e-05, affine=True)
        self.conv_act = nn.SiLU()
        self.conv_out = nn.Conv2d(320, 4, kernel_size=3, stride=1, padding=1)

    def forward(
        self, sample, timesteps, encoder_hidden_states, added_cond_kwargs, **kwargs
    ):
        # Implement the forward pass through the model
        timesteps = timesteps.expand(sample.shape[0])
        t_emb = self.time_proj(timesteps).to(dtype=sample.dtype)
        emb = self.time_embedding(t_emb)

        sample = self.conv_in(sample)

        # 3. down
        s0 = sample
        sample, [s1, s2, s3] = self.down_blocks[0](
            sample,
            temb=emb,
            encoder_hidden_states=encoder_hidden_states,
        )

        sample, [s4, s5, s6] = self.down_blocks[1](
            sample,
            temb=emb,
            encoder_hidden_states=encoder_hidden_states,
        )

        sample, [s7, s8, s9] = self.down_blocks[2](
            sample,
            temb=emb,
            encoder_hidden_states=encoder_hidden_states,
        )

        sample, [s10, s11] = self.down_blocks[3](
            sample,
            temb=emb,
        )

        # 4. mid
        sample = self.mid_block(
            sample, emb, encoder_hidden_states=encoder_hidden_states
        )

        # 5. up
        _, [us1, us2, us3] = self.up_blocks[0](
            hidden_states=sample,
            temb=emb,
            res_hidden_states_tuple=[s9, s10, s11],
        )

        _, [us4, us5, us6] = self.up_blocks[1](
            hidden_states=us3,
            temb=emb,
            res_hidden_states_tuple=[s6, s7, s8],
            encoder_hidden_states=encoder_hidden_states,
        )

        _, [us7, us8, us9] = self.up_blocks[2](
            hidden_states=us6,
            temb=emb,
            res_hidden_states_tuple=[s3, s4, s5],
            encoder_hidden_states=encoder_hidden_states,
        )

        _, [us10, us11, us12] = self.up_blocks[3](
            hidden_states=us9,
            temb=emb,
            res_hidden_states_tuple=[s0, s1, s2],
            encoder_hidden_states=encoder_hidden_states,
        )

        # 6. post-process
        sample = self.conv_norm_out(us12)
        sample = self.conv_act(sample)
        sample = self.conv_out(sample)

        return [sample]

### Feature Extraction

In [11]:
import torch
from torch import nn
import torch.nn.functional as F
import einops
from diffusers import DiffusionPipeline
from jaxtyping import Float, Int
from pydoc import locate
from typing import Literal


class SD21UNetFeatureExtractor(SD21UNetModel):
    def __init__(self):
        super().__init__()

    def forward(
        self, sample, timesteps, encoder_hidden_states, added_cond_kwargs, **kwargs
    ):
        timesteps = timesteps.expand(sample.shape[0])
        t_emb = self.time_proj(timesteps).to(dtype=sample.dtype)
        emb = self.time_embedding(t_emb)

        sample = self.conv_in(sample)

        # 3. down
        s0 = sample
        sample, [s1, s2, s3] = self.down_blocks[0](
            sample,
            temb=emb,
            encoder_hidden_states=encoder_hidden_states,
        )

        sample, [s4, s5, s6] = self.down_blocks[1](
            sample,
            temb=emb,
            encoder_hidden_states=encoder_hidden_states,
        )

        sample, [s7, s8, s9] = self.down_blocks[2](
            sample,
            temb=emb,
            encoder_hidden_states=encoder_hidden_states,
        )

        sample, [s10, s11] = self.down_blocks[3](
            sample,
            temb=emb,
        )

        # 4. mid
        sample_mid = self.mid_block(
            sample, emb, encoder_hidden_states=encoder_hidden_states
        )

        # 5. up
        _, [us1, us2, us3] = self.up_blocks[0](
            hidden_states=sample_mid,
            temb=emb,
            res_hidden_states_tuple=[s9, s10, s11],
        )

        _, [us4, us5, us6] = self.up_blocks[1](
            hidden_states=us3,
            temb=emb,
            res_hidden_states_tuple=[s6, s7, s8],
            encoder_hidden_states=encoder_hidden_states,
        )

        _, [us7, us8, us9] = self.up_blocks[2](
            hidden_states=us6,
            temb=emb,
            res_hidden_states_tuple=[s3, s4, s5],
            encoder_hidden_states=encoder_hidden_states,
        )

        _, [us10, us11, _] = self.up_blocks[3](
            hidden_states=us9,
            temb=emb,
            res_hidden_states_tuple=[s0, s1, s2],
            encoder_hidden_states=encoder_hidden_states,
        )

        return {
            "mid": sample_mid,
            "us1": us1,
            "us2": us2,
            "us3": us3,
            "us4": us4,
            "us5": us5,
            "us6": us6,
            "us7": us7,
            "us8": us8,
            "us9": us9,
            "us10": us10,
        }


class FeedForwardBlockCustom(FeedForwardBlock):
    def __init__(
        self,
        d_model: int,
        d_ff: int,
        d_cond_norm: int = None,
        norm_type: Literal["AdaRMS", "FiLM"] = "AdaRMS",
        use_gating: bool = True,
    ):
        super().__init__(d_model=d_model, d_ff=d_ff, d_cond_norm=d_cond_norm)
        if not use_gating:
            self.up_proj = LinearSwish(d_model, d_ff, bias=False)
        if norm_type == "FiLM":
            self.norm = FiLMNorm(d_model, d_cond_norm)


class FFNStack(nn.Module):
    def __init__(
        self,
        dim: int,
        depth: int,
        ffn_expansion: float,
        dim_cond: int,
        norm_type: Literal["AdaRMS", "FiLM"] = "AdaRMS",
        use_gating: bool = True,
    ) -> None:
        super().__init__()
        self.layers = nn.ModuleList(
            [
                FeedForwardBlockCustom(
                    d_model=dim,
                    d_ff=int(dim * ffn_expansion),
                    d_cond_norm=dim_cond,
                    norm_type=norm_type,
                    use_gating=use_gating,
                )
                for _ in range(depth)
            ]
        )

    def forward(self, x: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, cond_norm=cond)
        return x


class FiLMNorm(nn.Module):
    def __init__(self, features, cond_features):
        super().__init__()
        self.linear = Linear(cond_features, features * 2, bias=False)
        self.feature_dim = features

    def forward(self, x, cond):
        B, _, D = x.shape
        scale, shift = self.linear(cond).chunk(2, dim=-1)
        # broadcast scale and shift across all features
        scale = scale.view(B, 1, D)
        shift = scale.view(B, 1, D)
        return scale * x + shift


class LinearSwish(nn.Linear):
    def __init__(self, in_features, out_features, bias=True):
        super().__init__(in_features, out_features, bias=bias)

    def forward(self, x):
        return F.silu(super().forward(x))


class ArgSequential(
    nn.Module
):  # Utility class to enable instantiating nn.Sequential instances with Hydra
    def __init__(self, *layers) -> None:
        super().__init__()
        self.layers = nn.ModuleList(layers)

    def forward(self, x, *args, **kwargs):
        for layer in self.layers:
            x = layer(x, *args, **kwargs)
        return x


class StableFeatureAligner(nn.Module):
    def __init__(
        self,
        ae: nn.Module,
        mapping,
        adapter_layer_class: str,
        feature_dims: dict[str, int],
        feature_extractor_cls: str,
        sd_version: Literal["sd15", "sd21"],
        adapter_layer_params: dict = {},
        use_text_condition: bool = False,
        t_min: int = 1,
        t_max: int = 999,
        t_max_model: int = 999,
        num_t_stratification_bins: int = 3,
        alignment_loss: Literal["cossim", "mse", "l1"] = "cossim",
        train_unet: bool = True,
        train_adapter: bool = True,
        t_init: int = 261,
        learn_timestep: bool = False,
        val_dataset: torch.utils.data.Dataset | None = None,
        val_t: int = 261,
        val_feature_key: str = "us6",
        val_chunk_size: int = 10,
        use_adapters: bool = True,
    ):
        super().__init__()
        self.ae = ae
        self.sd_version = sd_version
        self.val_t = val_t
        self.val_feature_key = val_feature_key
        self.val_dataset = val_dataset
        self.val_chunk_size = val_chunk_size
        self.use_adapters = use_adapters

        if sd_version == "sd15":
            self.repo = "stable-diffusion-v1-5/stable-diffusion-v1-5"
        elif sd_version == "sd21":
            self.repo = "stabilityai/stable-diffusion-2-1"
        else:
            raise ValueError(f"Invalid SD version: {sd_version}")

        self.mapping = None
        if use_adapters:
            self.time_emb = FourierFeatures(1, mapping.width)
            self.time_in_proj = Linear(mapping.width, mapping.width, bias=False)
            self.mapping = MappingNetwork(
                mapping.depth, mapping.width, mapping.d_ff, dropout=mapping.dropout
            )
            self.mapping.compile()

        if use_adapters:
            self.adapters = nn.ModuleDict()
            for k, dim in feature_dims.items():
                self.adapters[k] = locate(adapter_layer_class)(
                    dim=dim, **adapter_layer_params
                )
                self.adapters[k].requires_grad_(train_adapter)

        self.unet_feature_extractor_base = locate(feature_extractor_cls)().cuda()
        self.pipe = DiffusionPipeline.from_pretrained(
            self.repo,
            torch_dtype=torch.bfloat16,
            use_safetensors=True,
        ).to("cuda")
        self.unet_feature_extractor_base.load_state_dict(self.pipe.unet.state_dict())
        self.unet_feature_extractor_base.eval()
        self.unet_feature_extractor_base.requires_grad_(False)
        self.unet_feature_extractor_base.compile()

        self.unet_feature_extractor_cleandift = locate(feature_extractor_cls)().cuda()
        self.unet_feature_extractor_cleandift.load_state_dict(
            {
                k: v.detach().clone()
                for k, v in self.unet_feature_extractor_base.state_dict().items()
            }
        )

        if train_unet or learn_timestep:
            self.unet_feature_extractor_cleandift.train()
        else:
            self.unet_feature_extractor_cleandift.eval()
        self.unet_feature_extractor_cleandift.requires_grad_(train_unet)
        self.unet_feature_extractor_cleandift.compile()

        self.use_text_condition = use_text_condition
        if self.use_text_condition:
            self.pipe.text_encoder.compile()
        else:
            with torch.no_grad():
                prompt_embeds_dict = self.get_prompt_embeds([""])
                self._empty_prompt_embeds = prompt_embeds_dict["prompt_embeds"]
                del self.pipe.text_encoder

        del self.pipe.unet, self.pipe.vae

        self.t_min = t_min
        self.t_max = t_max
        self.t_max_model = t_max_model
        self.num_t_stratification_bins = num_t_stratification_bins
        self.alignment_loss = alignment_loss
        self.timestep = nn.Parameter(
            torch.tensor(float(t_init), requires_grad=learn_timestep),
            requires_grad=learn_timestep,
        )

    def get_prompt_embeds(self, prompt: list[str]) -> dict[str, torch.Tensor | None]:
        self.prompt_embeds, _ = self.pipe.encode_prompt(
            prompt=prompt,
            device=torch.device("cuda"),
            num_images_per_prompt=1,
            do_classifier_free_guidance=False,
        )
        return {"prompt_embeds": self.prompt_embeds}

    def _get_unet_conds(
        self, prompts: list[str], device, dtype, N_T
    ) -> dict[str, torch.Tensor]:
        B = len(prompts)
        if self.use_text_condition:
            prompt_embeds_dict = self.get_prompt_embeds(prompts)
        else:
            prompt_embeds_dict = {
                "prompt_embeds": einops.repeat(
                    self._empty_prompt_embeds, "b ... -> (B b) ...", B=B
                )
            }

        unet_conds = {
            "encoder_hidden_states": einops.repeat(
                prompt_embeds_dict["prompt_embeds"], "B ... -> (B N_T) ...", N_T=N_T
            ).to(dtype=dtype, device=device),
            "added_cond_kwargs": {},
        }

        return unet_conds

    def forward(
        self, x: Float[torch.Tensor, "b c h w"], caption: list[str], **kwargs
    ) -> tuple[dict[str, torch.Tensor], dict[str, torch.Tensor]]:
        B, *_ = x.shape
        device = x.device
        t_range = self.t_max - self.t_min
        t_range_per_bin = t_range / self.num_t_stratification_bins
        t: Int[torch.Tensor, "B N_T"] = (
            self.t_min
            + torch.rand((B, self.num_t_stratification_bins), device=device)
            * t_range_per_bin
            + torch.arange(0, self.num_t_stratification_bins, device=device)[None, :]
            * t_range_per_bin
        ).long()
        B, N_T = t.shape

        with torch.no_grad():
            unet_conds = self._get_unet_conds(caption, device, x.dtype, N_T)
            x_0: Float[torch.Tensor, "(B N_T) ..."] = self.ae.encode(x)
            x_0 = einops.repeat(x_0, "B ... -> (B N_T) ...", N_T=N_T)
            _, *latent_shape = x_0.shape
            noise_sample = torch.randn(
                (B * N_T, *latent_shape), device=device, dtype=x.dtype
            )

            x_t: Float[torch.Tensor, "(B N_T) ..."] = self.pipe.scheduler.add_noise(
                x_0,
                noise_sample,
                einops.rearrange(t, "B N_T -> (B N_T)"),
            )

            feats_base: dict[str, Float[torch.Tensor, "B N_T ..."]] = {
                k: einops.rearrange(v, "(B N_T) D H W -> B N_T (H W) D", B=B, N_T=N_T)
                for k, v in self.unet_feature_extractor_base(
                    x_t,
                    einops.rearrange(t, "B N_T -> (B N_T)"),
                    **unet_conds,
                ).items()
            }

        feats_cleandift: dict[str, Float[torch.Tensor, "B N_T ..."]] = {
            k: einops.rearrange(v, "(B N_T) D H W -> B N_T (H W) D", N_T=N_T)
            for k, v in self.unet_feature_extractor_cleandift(
                x_0,
                einops.rearrange(
                    torch.ones_like(t) * self.timestep, "B N_T -> (B N_T)"
                ),
                **unet_conds,
            ).items()
        }

        if self.use_adapters:
            # time conditioning for adapters
            if not self.mapping is None:
                map_cond: Float[torch.Tensor, "(B N_T) ..."] = self.mapping(
                    self.time_in_proj(
                        self.time_emb(
                            einops.rearrange(t, "B N_T -> (B N_T) 1").to(
                                dtype=x.dtype, device=device
                            )
                            / self.t_max_model
                        )
                    )
                )

            feats_cleandift: dict[str, Float[torch.Tensor, "B N_T ..."]] = {
                k: einops.rearrange(
                    self.adapters[k](
                        einops.rearrange(v, "B N_T ... -> (B N_T) ..."), cond=map_cond
                    ),
                    "(B N_T) ... -> B N_T ...",
                    B=B,
                    N_T=N_T,
                )
                for k, v in feats_cleandift.items()
            }

        if self.alignment_loss == "mse":
            return {
                f"mse_{k}": F.mse_loss(feats_cleandift[k], v.detach())
                for k, v in feats_base.items()
            }
        elif self.alignment_loss == "l1":
            return {
                f"l1_{k}": F.l1_loss(feats_cleandift[k], v.detach())
                for k, v in feats_base.items()
            }
        elif self.alignment_loss == "cossim":
            return {
                f"neg_cossim_{k}": -F.cosine_similarity(
                    feats_cleandift[k], v.detach(), dim=-1
                ).mean()
                for k, v in feats_base.items()
            }
        else:
            raise ValueError(f"Invalid alignment loss type: {self.alignment_loss}")

    @torch.no_grad()
    def get_features(
        self,
        x: Float[torch.Tensor, "b c h w"],
        caption: list[str] | None,
        t: Int[torch.Tensor, "b"] | None,
        feat_key: str,
        use_base_model: bool = False,
        input_pure_noise: bool = False,
        eps: torch.Tensor = None,
    ) -> Float[torch.Tensor, "b d h' w'"]:
        if use_base_model:
            assert not t is None
            B, *_ = x.shape

            if caption is None:
                caption = [""] * B

            unet_conds = self._get_unet_conds(caption, x.device, x.dtype, 1)
            x_0 = self.ae.encode(x)
            eps = torch.randn_like(x_0) if eps is None else eps
            if input_pure_noise:
                assert torch.allclose(
                    t, torch.full_like(t, 999)
                ), "Sanity check. Pure noise means that no x_t is given to the U-Net, just pure noise (eps)."
                x_t = eps
            else:
                x_t = self.pipe.scheduler.add_noise(x_0, eps, t)

            if feat_key is None:
                feats = self.unet_feature_extractor_base(x_t, t, **unet_conds)
            else:
                feats = self.unet_feature_extractor_base(x_t, t, **unet_conds)[feat_key]
            return feats
        else:
            (B, *_), device = x.shape, x.device

            if caption is None:
                caption = [""] * B

            unet_conds = self._get_unet_conds(caption, device, x.dtype, 1)
            x_0 = self.ae.encode(x)

            feats = self.unet_feature_extractor_cleandift(
                x_0,
                torch.ones((B,), device=device, dtype=self.timestep.dtype)
                * self.timestep,
                **unet_conds,
            )

            if feat_key is not None:
                feats = feats[feat_key]

            feats = einops.rearrange(
                feats,
                "B D H W -> B H W D",
            )
            if t is None:
                return einops.rearrange(feats, "B H W D -> B D H W")
            else:
                assert (
                    self.use_adapters
                ), "Adapters must be enabled to use t conditioning on cleandift model"
                map_cond: Float[torch.Tensor, "B ..."] = self.mapping(
                    self.time_in_proj(
                        self.time_emb(
                            t[:, None].to(dtype=x.dtype, device=device)
                            / self.t_max_model
                        )
                    )
                )
                if feat_key is not None:
                    return einops.rearrange(
                        self.adapters[feat_key](feats, cond=map_cond),
                        "B H W D -> B D H W",
                    )
                else:
                    return {
                        key: einops.rearrange(
                            self.adapters[key](feats[key], cond=map_cond),
                            "B H W D -> B D H W",
                        )
                        for key in feats.keys()
                    }

### Data Loader

In [12]:
import copy
import json
import os
import torch
import torch.utils.data as data
from PIL import Image
from torchvision.transforms.functional import to_tensor
from torchvision import transforms
from typing import Tuple


def load_image(path: str, img_size: int):
    image = Image.open(path).convert("RGB")
    resize_transform = transforms.Resize((img_size, img_size))
    image = resize_transform(image)
    image = to_tensor(image) * 2 - 1
    return image


class DummyDataset(data.Dataset):
    def __init__(self, dataset_dir: str, img_size: int = 512, train: bool = True):
        self.dataset_dir = dataset_dir
        self.img_size = img_size
        self.data = []

        jpg_files = [f for f in os.listdir(dataset_dir) if f.endswith(".jpg")]
        for img_path in jpg_files:
            json_path = os.path.join(
                dataset_dir, os.path.splitext(img_path)[0] + ".json"
            )
            assert os.path.exists(json_path)
            with open(json_path, "r") as json_file:
                json_dict = json.load(json_file)
            assert "caption" in json_dict.keys()
            self.data.append(
                {
                    "img_path": os.path.join(dataset_dir, img_path),
                    "caption": json_dict["caption"],
                }
            )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        sample = {}
        # Load image
        sample["x"] = load_image(item["img_path"], img_size=self.img_size)
        sample["caption"] = item["caption"]
        return sample


class DataModule:
    def __init__(self, dataset_dir: str, batch_size: int = 1, img_size: int = 512):
        self.batch_size = batch_size

        train_dataset = DummyDataset(
            dataset_dir=dataset_dir, train=True, img_size=img_size
        )
        self.train_loader = torch.utils.data.DataLoader(
            train_dataset, batch_size=batch_size
        )

        val_dataset = DummyDataset(
            dataset_dir=dataset_dir, train=False, img_size=img_size
        )
        self.val_loader = torch.utils.data.DataLoader(
            val_dataset, batch_size=batch_size
        )

    def train_dataloader(self):
        return self.train_loader

    def val_dataloader(self):
        return self.val_loader

### Autoencoder with KL Loss Divergence

In [13]:
import copy
import json
import os
import torch
import torch.utils.data as data
from PIL import Image
from torchvision.transforms.functional import to_tensor
from torchvision import transforms
from typing import Tuple


def load_image(path: str, img_size: int):
    image = Image.open(path).convert("RGB")
    resize_transform = transforms.Resize((img_size, img_size))
    image = resize_transform(image)
    image = to_tensor(image) * 2 - 1
    return image


class DummyDataset(data.Dataset):
    def __init__(self, dataset_dir: str, img_size: int = 512, train: bool = True):
        self.dataset_dir = dataset_dir
        self.img_size = img_size
        self.data = []

        jpg_files = [f for f in os.listdir(dataset_dir) if f.endswith(".jpg")]
        for img_path in jpg_files:
            json_path = os.path.join(
                dataset_dir, os.path.splitext(img_path)[0] + ".json"
            )
            assert os.path.exists(json_path)
            with open(json_path, "r") as json_file:
                json_dict = json.load(json_file)
            assert "caption" in json_dict.keys()
            self.data.append(
                {
                    "img_path": os.path.join(dataset_dir, img_path),
                    "caption": json_dict["caption"],
                }
            )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        sample = {}
        # Load image
        sample["x"] = load_image(item["img_path"], img_size=self.img_size)
        sample["caption"] = item["caption"]
        return sample


class DataModule:
    def __init__(self, dataset_dir: str, batch_size: int = 1, img_size: int = 512):
        self.batch_size = batch_size

        train_dataset = DummyDataset(
            dataset_dir=dataset_dir, train=True, img_size=img_size
        )
        self.train_loader = torch.utils.data.DataLoader(
            train_dataset, batch_size=batch_size
        )

        val_dataset = DummyDataset(
            dataset_dir=dataset_dir, train=False, img_size=img_size
        )
        self.val_loader = torch.utils.data.DataLoader(
            val_dataset, batch_size=batch_size
        )

    def train_dataloader(self):
        return self.train_loader

    def val_dataloader(self):
        return self.val_loader

### Depth Estimation

- Remove YAML configs using hydra

In [16]:
from typing import Literal
from omegaconf import OmegaConf
import hydra
import torch
from torch import nn
from jaxtyping import Float
import torch.nn.functional as F


class SigLoss(nn.Module):
    """SigLoss.

        This follows `AdaBins <https://arxiv.org/abs/2011.14141>`_.

    Args:
        valid_mask (bool): Whether filter invalid gt (gt > 0). Default: True.
        loss_weight (float): Weight of the loss. Default: 1.0.
        max_depth (int): When filtering invalid gt, set a max threshold. Default: None.
        warm_up (bool): A simple warm up stage to help convergence. Default: False.
        warm_iter (int): The number of warm up stage. Default: 100.

    Adapted from: https://github.com/facebookresearch/dinov2/blob/main/dinov2/eval/depth/models/losses/sigloss.py
    """

    def __init__(
        self,
        valid_mask=True,
        loss_weight=1.0,
        max_depth=None,
        warm_up=False,
        warm_iter=100,
        loss_name="sigloss",
    ):
        super(SigLoss, self).__init__()
        self.valid_mask = valid_mask
        self.loss_weight = loss_weight
        self.max_depth = max_depth
        self.loss_name = loss_name

        self.eps = 0.001

        self.warm_up = warm_up
        self.warm_iter = warm_iter
        self.warm_up_counter = 0

    def sigloss(self, input, target):
        if self.valid_mask:
            valid_mask = target > 0
            if self.max_depth is not None:
                valid_mask = torch.logical_and(target > 0, target <= self.max_depth)
            input = input[valid_mask]
            target = target[valid_mask]

        if self.warm_up:
            if self.warm_up_counter < self.warm_iter:
                g = torch.log(input + self.eps) - torch.log(target + self.eps)
                g = 0.15 * torch.pow(torch.mean(g), 2)
                self.warm_up_counter += 1
                return torch.sqrt(g)

        g = torch.log(input + self.eps) - torch.log(target + self.eps)
        Dg = torch.var(g) + 0.15 * torch.pow(torch.mean(g), 2)
        return torch.sqrt(Dg)

    def forward(self, depth_pred, depth_gt):
        """Forward function."""

        loss_depth = self.loss_weight * self.sigloss(depth_pred, depth_gt)
        return loss_depth


class DepthPred(torch.nn.Module):
    """
    Adapted from:
        - https://github.com/facebookresearch/dinov2/blob/main/dinov2/eval/depth/models/decode_heads/decode_head.py
        - https://github.com/facebookresearch/dinov2/blob/main/dinov2/eval/depth/models/decode_heads/linear_head.py
    """

    def __init__(
        self,
        model_config_path: str,
        channels: int,
        loss: nn.Module,
        diffusion_image_size: int,
        classify=True,
        n_bins=256,
        min_depth=0.1,
        max_depth=10.0,
        bins_strategy="UD",
        norm_strategy="linear",
        scale_up=False,
        use_base_model_features=False,
        base_model_timestep: None | int = None,
        adapter_timestep: int | None = None,
        extraction_layer="us6",
        interpolate_features: Literal["DINO", "FULL", "NONE"] = "NONE",
        n_vis_samples: int = 0,
    ):
        super().__init__()
        self.loss = loss
        self.classify = classify
        self.n_bins = n_bins
        self.min_depth = min_depth
        self.max_depth = max_depth
        self.scale_up = scale_up
        self.use_base_model_features = use_base_model_features
        self.base_model_timestep = base_model_timestep
        self.diffusion_image_size = diffusion_image_size
        self.extraction_layer = extraction_layer
        self.interpolate_features = interpolate_features
        self.adapter_timestep = adapter_timestep
        self.n_vis_samples = n_vis_samples

        if use_base_model_features:
            assert (
                base_model_timestep is not None
            ), "Need to provide base_model_timestep if using base model features"

        cfg_model = OmegaConf.load(model_config_path)
        OmegaConf.resolve(cfg_model)
        self.feature_extractor = hydra.utils.instantiate(cfg_model).model

        self.feature_extractor.requires_grad_(False)
        self.feature_extractor.eval()

        if self.classify:
            assert bins_strategy in ["UD", "SID"], "Support bins_strategy: UD, SID"
            assert norm_strategy in [
                "linear",
                "softmax",
                "sigmoid",
            ], "Support norm_strategy: linear, softmax, sigmoid"

            self.bins_strategy = bins_strategy
            self.norm_strategy = norm_strategy
            self.conv_depth = nn.Conv2d(
                channels, n_bins, kernel_size=1, padding=0, stride=1
            )
        else:
            self.conv_depth = nn.Conv2d(channels, 1, kernel_size=1, padding=0, stride=1)

    def depth_pred(
        self, feat: Float[torch.Tensor, "B C H W"]
    ) -> Float[torch.Tensor, "B 1 H W"]:
        """Prediction each pixel."""
        if self.classify:
            logit = self.conv_depth(feat)

            if self.bins_strategy == "UD":
                bins = torch.linspace(
                    self.min_depth,
                    self.max_depth,
                    self.n_bins,
                    device=feat.device,
                    dtype=feat.dtype,
                )
            elif self.bins_strategy == "SID":
                bins = torch.logspace(
                    self.min_depth,
                    self.max_depth,
                    self.n_bins,
                    device=feat.device,
                    dtype=feat.dtype,
                )

            # following Adabins, default linear
            if self.norm_strategy == "linear":
                logit = torch.relu(logit)
                eps = 0.1
                logit = logit + eps
                logit = logit / logit.sum(dim=1, keepdim=True)
            elif self.norm_strategy == "softmax":
                logit = torch.softmax(logit, dim=1)
            elif self.norm_strategy == "sigmoid":
                logit = torch.sigmoid(logit)
                logit = logit / logit.sum(dim=1, keepdim=True)

            output = torch.einsum("ikmn,k->imn", [logit, bins]).unsqueeze(dim=1)

        else:
            if self.scale_up:
                output = torch.sigmoid(self.conv_depth(feat)) * self.max_depth
            else:
                output = torch.relu(self.conv_depth(feat)) + self.min_depth

        return output

    def forward(self, img, depth_gt, *args, **kwargs):
        depth_target = depth_gt.data[0].cuda().bfloat16()
        image = img.data[0].cuda().bfloat16()

        depth_pred = self.predict(image)

        if self.interpolate_features != "FULL":
            H, W = depth_pred.shape[-2:]
            depth_target = F.interpolate(
                depth_target, (H, W), mode="bilinear", align_corners=False
            )

        return self.loss(depth_pred, depth_target)

    def predict(self, image):

        B, C, H, W = image.shape
        image = F.interpolate(
            image,
            size=(self.diffusion_image_size, self.diffusion_image_size),
            mode="bilinear",
            align_corners=False,
        )

        with torch.no_grad():
            timestep = None
            if self.use_base_model_features:
                timestep = torch.tensor(
                    [self.base_model_timestep] * B, device=image.device
                )
            elif self.adapter_timestep is not None:
                timestep = torch.tensor(
                    [self.adapter_timestep] * B, device=image.device
                )
            features = self.feature_extractor.get_features(
                image,
                ["A photo of a room"] * B,
                timestep,
                self.extraction_layer,
                use_base_model=self.use_base_model_features,
            )

            if self.interpolate_features == "FULL":
                features = F.interpolate(
                    features, size=(H, W), mode="bilinear", align_corners=False
                )
            elif self.interpolate_features == "DINO":
                H, W = features.shape[-2:]
                features = F.interpolate(
                    features, size=(H * 4, W * 4), mode="bilinear", align_corners=False
                )

        depth_pred = self.depth_pred(features)

        return depth_pred

ModuleNotFoundError: No module named 'hydra'

### Training Loop

In [19]:
import argparse
import hydra
import logging
import os
import torch
from typing import Optional
from omegaconf import DictConfig, OmegaConf
from src.utils import set_seed, dict_to
from transformers import get_scheduler
from tqdm.auto import tqdm


@hydra.main(
    config_path="configs", config_name="sd15_feature_extractor", version_base="1.1"
)
def main(cfg: DictConfig):
    OmegaConf.resolve(cfg)
    set_seed(cfg.seed)
    logger = logging.getLogger(f"{__name__}")
    device = torch.device("cuda:0")

    # Load model
    cfg = hydra.utils.instantiate(cfg)
    model = cfg.model.to(device)
    model.train()

    data = cfg.data
    dataloader_train = data.train_dataloader()

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    lr_scheduler = get_scheduler(
        name=cfg.lr_scheduler.name,
        optimizer=optimizer,
        num_warmup_steps=cfg.lr_scheduler.num_warmup_steps,
        num_training_steps=cfg.lr_scheduler.num_training_steps,
        scheduler_specific_kwargs=OmegaConf.to_container(
            cfg.lr_scheduler.scheduler_specific_kwargs
        ),
    )

    i_epoch = -1
    stop = False
    max_steps: Optional[int] = cfg.max_steps

    val_freq: Optional[int] = cfg.val_freq
    if not val_freq is None:
        dataloader_val = data.val_dataloader()
    max_val_steps: Optional[int] = cfg.max_val_steps
    checkpoint_freq: Optional[int] = cfg.checkpoint_freq
    checkpoint_dir: str = cfg.checkpoint_dir
    os.makedirs(checkpoint_dir, exist_ok=True)

    grad_accum_steps = cfg.grad_accum_steps
    print(f"grad_accum_steps={grad_accum_steps}")

    step = 0

    while not stop:  # Epochs
        i_epoch += 1
        for batch in (
            pbar := tqdm(dataloader_train, desc=f"Optimizing (Epoch {i_epoch + 1})")
        ):
            loss_sum = 0
            for accum_step in range(grad_accum_steps):
                losses = model(**dict_to(batch, device=device))
                loss = sum(v.mean() for v in losses.values())
                loss.backward()
                loss_sum += float(loss.detach().item())
                pbar.set_postfix({"loss": loss_sum / (accum_step + 1)})

            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

            if not val_freq is None and step % val_freq == 0:
                model.eval()

                with torch.no_grad():
                    val_losses_accumulated = []
                    for i, val_batch in enumerate(
                        tqdm(dataloader_val, desc=f"Validating", total=max_val_steps)
                    ):
                        val_losses = model(**dict_to(val_batch, device=device))
                        val_loss = sum(v.mean() for v in val_losses.values())
                        val_losses_accumulated.append((val_loss).cpu().item())

                        if max_val_steps is not None and i + 1 >= max_val_steps:
                            break

                    val_loss = sum(val_losses_accumulated) / len(val_losses_accumulated)
                    logger.info(f"Validation loss: {val_loss}")

                # put model into train mode
                model.train()

            if not checkpoint_freq is None and (step + 1) % checkpoint_freq == 0:
                checkpoint_path = os.path.join(checkpoint_dir, f"step_{(step + 1)}.pth")
                torch.save(model.state_dict(), checkpoint_path)
                logger.info(f"Saved checkpoint to {checkpoint_path}")

            if not max_steps is None and step == max_steps:
                stop = True
                break

            step += 1


if __name__ == "__main__":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch._dynamo.config.cache_size_limit = max(
        64, torch._dynamo.config.cache_size_limit
    )
    main()

ModuleNotFoundError: No module named 'hydra'

## References

- [CleanDIFT: Diffusion Features without Noise](https://compvis.github.io/cleandift/) 
- [Understanding Seeds in AI: The Key to Reproducibility and Creativity](https://medium.com/@nikunj.vaghasiya2050/understanding-seeds-in-ai-the-key-to-reproducibility-and-creativity-edcfd3bf649c)
- [What is Residual Connection?](https://medium.com/data-science/what-is-residual-connection-efb07cab0d55)
- [What is a conditional vector?](https://stats.stackexchange.com/questions/338178/gans-stackgan-paper-what-is-a-conditioning-vector)
- [Feedforward Neural Network](https://www.sciencedirect.com/topics/computer-science/feedforward-neural-network)
- [AdaNorm: Adaptive Gradient Norm Correction based on Optimizer for CNNs](https://arxiv.org/abs/2210.06364)
- [Root Mean Square Normalization (RMSNorm)](https://arxiv.org/abs/1910.07467)
- [SwiGLU Activation Function](https://abdulkaderhelwan.medium.com/swiglu-activation-function-77627e0b2b52)
- [Random Fourier Features](https://gregorygundersen.com/blog/2019/12/23/random-fourier-features/)